<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# Datasets de gran volumen — Yelp Open Dataset


> *¿Qué hacemos cuando un dataset no entra en memoria? En esta notebook procesamos el Yelp Open Dataset — 8 GB de datos reales — usando técnicas de lectura por lotes, proyección de columnas y filtrado en carga.*

---

### El dataset

El **Yelp Open Dataset** es un conjunto de datos reales publicado por Yelp para uso académico.
Se descarga sin necesidad de registrarse en [yelp.com/dataset](https://www.yelp.com/dataset).

| Archivo | Descripción | Tamaño | Filas |
|---------|-------------|--------|-------|
| `yelp_academic_dataset_business.json` | Negocios: nombre, ciudad, rating, categorías | ~120 MB | ~150.000 |
| `yelp_academic_dataset_review.json` | Reviews: texto, rating, fecha | ~5 GB | ~7.000.000 |
| `yelp_academic_dataset_user.json` | Usuarios: actividad, seguidores | ~3 GB | ~2.000.000 |
| `yelp_academic_dataset_tip.json` | Tips cortos de usuarios | ~250 MB | ~1.300.000 |
| `yelp_academic_dataset_checkin.json` | Check-ins por negocio | ~290 MB | ~130.000 |

En esta notebook trabajamos con `business.json` y `review.json`.

---
## Configuración inicial

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── Ajustar según dónde descomprimiste el dataset ────────────────
DATA_DIR = "./datasets/Yelp_JSON/yelp_dataset"
# ─────────────────────────────────────────────────────────────────

BUSINESS_PATH = os.path.join(DATA_DIR, "yelp_academic_dataset_business.json")
REVIEW_PATH   = os.path.join(DATA_DIR, "yelp_academic_dataset_review.json")

---
### 1. El problema del tamaño

Antes de leer el archivo, veamos cuánto pesa.

In [15]:
def tamaño_legible(path):
    """Devuelve el tamaño de un archivo en MB o GB."""
    bytes_ = os.path.getsize(path)
    if bytes_ >= 1_000_000_000:
        return f"{bytes_ / 1_000_000_000:.2f} GB"
    return f"{bytes_ / 1_000_000:.1f} MB"

print(f"business.json : {tamaño_legible(BUSINESS_PATH)}")
print(f"review.json   : {tamaño_legible(REVIEW_PATH)}")

business.json : 118.9 MB
review.json   : 5.34 GB


### ¿Por qué no podemos hacer `pd.read_json('business.json')`?

```python
df = pd.read_json(BUSINESS_PATH, lines=True)  # ← NO hacer esto con review.json
```

Para leer un archivo JSON, pandas necesita **cargarlo completo en RAM** antes de construir el DataFrame.

```
Archivo en disco      RAM
─────────────────     ──────────────────────────────────
review.json           JSON crudo (5 GB)
5 GB                + DataFrame pandas (~ 8–10 GB)
                    ─────────────────────────────────────
                      ~15 GB simultáneos en memoria
```

Una computadora con 8 GB de RAM no puede con eso. Incluso con 16 GB, el sistema operativo y otras aplicaciones ya consumen varios GB.

**La solución**: leer el archivo en **lotes** (*chunks*) — procesar una porción a la vez, extraer lo que necesitamos, y descartar el resto.

---
### 2. Lectura por lotes con `chunksize`

Con `chunksize=N`, `pd.read_json()` no devuelve un DataFrame — devuelve un **iterador**.
Cada iteración entrega un DataFrame de N filas, lo procesamos, y lo descartamos antes de leer el siguiente.

```
Archivo (150.000 filas)
│
├── Chunk 1: filas    0 –  9.999  → procesar → acumular resultado → liberar
├── Chunk 2: filas 10.000 – 19.999 → procesar → acumular resultado → liberar
├── ...
└── Chunk 15: filas 140.000 – 149.999 → procesar → acumular resultado → liberar
                                                              │
                                                    pd.concat(resultados)
```

En memoria solo existe **un chunk a la vez**, no el archivo completo.

In [17]:
# Demostración: ver cómo se comporta el iterador
CHUNK_SIZE = 10_000

lector = pd.read_json(BUSINESS_PATH, lines=True, chunksize=CHUNK_SIZE)

# El lector es un iterador — no leyó nada todavía
print(f"Tipo del lector: {type(lector)}")
print()

# Pedimos el primer chunk
primer_chunk = next(lector)
print(f"Tipo del chunk:  {type(primer_chunk)}")
print(f"Filas:           {len(primer_chunk)}")
print(f"Columnas ({len(primer_chunk.columns)}): {list(primer_chunk.columns)}")

Tipo del lector: <class 'pandas.api.typing.JsonReader'>

Tipo del chunk:  <class 'pandas.DataFrame'>
Filas:           10000
Columnas (14): ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']


---
### 3. Proyección — cargar solo las columnas necesarias

El archivo tiene 14 columnas. Muchas no las necesitamos (`attributes`, `hours` son dicts muy anidados que ocupan mucha memoria).

**Proyectar** significa seleccionar solo las columnas útiles *dentro del chunk*, antes de acumularlo.
Es el equivalente del `SELECT col1, col2` en SQL — no traer todo si no lo vas a usar.

In [18]:
# Columnas que necesitamos para el análisis
COLS_BUSINESS = ['business_id', 'name', 'city', 'state', 'stars', 'review_count', 'categories', 'is_open']

# Comparar uso de memoria: chunk completo vs. chunk proyectado
chunk_completo   = primer_chunk
chunk_proyectado = primer_chunk[COLS_BUSINESS]

mem_completo   = chunk_completo.memory_usage(deep=True).sum() / 1024**2
mem_proyectado = chunk_proyectado.memory_usage(deep=True).sum() / 1024**2

print(f"Chunk completo   ({len(chunk_completo.columns)} columnas): {mem_completo:.1f} MB")
print(f"Chunk proyectado  ({len(chunk_proyectado.columns)} columnas): {mem_proyectado:.1f} MB")
print(f"Reducción: {(1 - mem_proyectado/mem_completo)*100:.0f}%")

Chunk completo   (14 columnas): 7.2 MB
Chunk proyectado  (8 columnas): 1.7 MB
Reducción: 77%


### Optimización de tipos

Proyectar columnas reduce el ancho del DataFrame. Pero también podemos reducir el **peso de cada columna** eligiendo el tipo de dato más chico que alcance para representar los valores.

`stars` va de 1.0 a 5.0 — solo necesita medio punto de precisión.  
Pandas lo lee como `float64` (8 bytes por valor) por default, pero `float32` (4 bytes) es más que suficiente.  
Si ni siquiera queremos decimales, `int8` (1 byte) alcanza para valores entre -128 y 127.

| Tipo | Bytes | Rango |
|------|-------|-------|
| `float64` | 8 | ±1.8 × 10³⁰⁸ |
| `float32` | 4 | ±3.4 × 10³⁸ |
| `int8`    | 1 | -128 a 127 |

**Regla práctica:** el rango del dato determina el tipo mínimo necesario.

En Python, el comportamiento numérico se divide en dos mundos. 

Por un lado, los números enteros (int) tienen precisión arbitraria, lo que significa que no tienen un límite fijo de tamaño y pueden crecer tanto como la memoria RAM lo permita, a diferencia de lenguajes tradicionales. 

Por otro lado, los números decimales (float) estándar sí tienen un límite estricto (equivalente a float64), operando con un rango máximo aproximado de $\pm 1.8 \times 10^{308}$ antes de desbordarse o convertirse en infinito.

Aunque Python maneja las variables estándar de forma automática, tipos de datos acotados como int8 o float32 cobran importancia al trabajar con librerías de análisis de datos y computación científica como NumPy o Pandas.

In [19]:
chunk_completo.dtypes

business_id         str
name                str
address             str
city                str
state               str
postal_code         str
latitude        float64
longitude       float64
stars           float64
review_count      int64
is_open           int64
attributes       object
categories          str
hours            object
dtype: object

In [24]:
# Comparar memoria de 'stars' antes y después de cambiar el tipo
# Trabajamos sobre el chunk proyectado que ya tenemos en memoria

col_antes = chunk_proyectado['stars']
col_float32 = col_antes.astype('float32')
col_int8    = col_antes.astype('int8')

n = len(col_antes)

mem_antes   = col_antes.memory_usage(deep=True)
mem_float32 = col_float32.memory_usage(deep=True)
mem_int8    = col_int8.memory_usage(deep=True)

print(f"Columna 'stars' — {n:,} filas\n")
print(f"  float64 (default) : {mem_antes:>8,} bytes  ({mem_antes/1024:.1f} KB)")
print(f"  float32           : {mem_float32:>8,} bytes  ({mem_float32/1024:.1f} KB)  — {(1 - mem_float32/mem_antes)*100:.0f}% menos")
print(f"  int8              : {mem_int8:>8,} bytes  ({mem_int8/1024:.1f} KB)  — {(1 - mem_int8/mem_antes)*100:.0f}% menos")
print()

cantidad_filas_extrapoladas=10_000_000
factor =  cantidad_filas_extrapoladas/ n

print(f"Extrapolado a {cantidad_filas_extrapoladas} de filas (review.json completo):")

print(f"  float64 : {mem_antes   * factor / 1024**2:.1f} MB")
print(f"  float32 : {mem_float32 * factor / 1024**2:.1f} MB")
print(f"  int8    : {mem_int8    * factor / 1024**2:.1f} MB")

Columna 'stars' — 10,000 filas

  float64 (default) :   80,132 bytes  (78.3 KB)
  float32           :   40,132 bytes  (39.2 KB)  — 50% menos
  int8              :   10,132 bytes  (9.9 KB)  — 87% menos

Extrapolado a 10000000 de filas (review.json completo):
  float64 : 76.4 MB
  float32 : 38.3 MB
  int8    : 9.7 MB


---
### 4. Filtrado en carga

Además de proyectar columnas, podemos **filtrar filas** dentro del chunk antes de acumularlas.
Así solo guardamos lo que realmente necesitamos.

En pandas, aplicamos ese filtro a cada chunk antes de acumularlo.

In [25]:
# Demostración con un chunk: cuántas filas pasan el filtro
chunk = primer_chunk[COLS_BUSINESS].copy()

mascara = (chunk['state'] == 'FL') & (chunk['is_open'] == 1)
chunk_filtrado = chunk[mascara]

print(f"Filas en el chunk:          {len(chunk):>6}")
print(f"Filas después del filtro:   {len(chunk_filtrado):>6}")
print(f"Porcentaje que pasa:        {len(chunk_filtrado)/len(chunk)*100:.1f}%")

Filas en el chunk:           10000
Filas después del filtro:     1403
Porcentaje que pasa:        14.0%


---
### 5. Lectura completa de `business.json`

Ahora aplicamos todo junto: iteramos sobre todos los chunks, proyectamos y filtramos en cada uno, y acumulamos solo los negocios de Florida que están abiertos.

In [ ]:
COLS_BUSINESS = ['business_id', 'name', 'city', 'state', 'stars', 'review_count', 'categories', 'is_open']
CHUNK_SIZE    = 10_000

acumulado = []
total_leidas = 0

for chunk in pd.read_json(BUSINESS_PATH, lines=True, chunksize=CHUNK_SIZE):
    total_leidas += len(chunk)

    # 1. Proyección: solo las columnas que necesitamos
    # ==================================================================================================================================================
    # NOTA DE MEMORIA: 
    # Al reasignar 'chunk = chunk[...]', creamos un nuevo DataFrame más pequeño  y la variable 'chunk' apunta ahora a este nuevo objeto. 
    # El bloque gigante original pierde sus referencias, lo que permite que Python libere esa memoria RAM automáticamente.
    # ==================================================================================================================================================
    chunk = chunk[COLS_BUSINESS]

    # 2. Filtrado: solo Florida, solo negocios abiertos
    chunk = chunk[(chunk['state'] == 'FL') & (chunk['is_open'] == 1)]

    if len(chunk) > 0:
        acumulado.append(chunk)

df_business = pd.concat(acumulado, ignore_index=True)

# Liberar la lista de chunks — ya no la necesitamos
# Todo el contenido quedó consolidado en df_business
del acumulado

print(f"Filas totales en el archivo:  {total_leidas:>8,}")
print(f"Negocios de FL (abiertos):    {len(df_business):>8,}")
print(f"Reducción:                    {(1 - len(df_business)/total_leidas)*100:.1f}%")
print()
df_business.head()

Filas totales en el archivo:   150,346
Negocios de FL (abiertos):      21,540
Reducción:                    85.7%



,business_id,name,city,state,stars,review_count,categories,is_open
0,qkRM_2X51Yqxk3btlwAQIg,Temple Beth-El,St. Petersburg,FL,3.5,5,"Synagogues, Religious Organizations",1
1,UJsufbvfyfONHeWdvAHKjA,Marshalls,Land O' Lakes,FL,3.5,6,"Department Stores, Shopping, Fashion",1
2,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,Tampa Bay,FL,4.0,10,"Vietnamese, Food, Restaurants, Food Trucks",1
3,jaxMSoInw8Poo3XeMJt8lQ,Adams Dental,Clearwater,FL,5.0,10,"General Dentistry, Dentists, Health & Medical,...",1
4,0qNpTGTcqPwOLi2hADx4Xw,Charlie's Market,Tampa,FL,3.0,9,"Food, Grocery, Convenience Stores",1


In [27]:
# Memoria final del DataFrame resultante
mem_mb = df_business.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria del DataFrame final: {mem_mb:.1f} MB")
print(f"Ciudades únicas en FL:       {df_business['city'].nunique()}")
print()
print("Top 10 ciudades por cantidad de negocios:")
df_business['city'].value_counts().head(10)

Memoria del DataFrame final: 3.7 MB
Ciudades únicas en FL:       219

Top 10 ciudades por cantidad de negocios:


city
Tampa               7217
Clearwater          1815
Saint Petersburg    1367
St. Petersburg       916
Brandon              848
Largo                827
Palm Harbor          537
New Port Richey      515
Riverview            509
Lutz                 500
Name: count, dtype: int64

---
## 6. Lectura de `review.json`

El archivo de reviews tiene **7 millones de filas y ~5 GB**. Ahora el filtrado es más interesante: no filtramos por un valor fijo,
sino por si el `business_id` de la review pertenece a un negocio de Florida.

Para eso necesitamos el **set de IDs** de Florida que ya construimos.

In [ ]:
# Set de business_id de Florida — lo usaremos para filtrar reviews
# Usamos set (no lista)
# ================================================================================================================================================
# NOTA DE RENDIMIENTO: Usamos set() en lugar de .unique() por dos razones:
# 1. Elimina duplicados automáticamente (igual que unique()).
# 2. Construye una tabla hash interna que permite hacer búsquedas (operadores 'in') con velocidad constante, optimizando el filtrado de reviews.
# ================================================================================================================================================
ids_florida = set(df_business['business_id'])

print(f"IDs de negocios FL en memoria: {len(ids_florida):,}")
print(f"Ejemplo de ID:                 {next(iter(ids_florida))}")

IDs de negocios FL en memoria: 21,540
Ejemplo de ID:                 a6md73H4aeJszwSBIcu3dQ


### Liberar memoria después del `concat`

Durante el loop, cada chunk filtrado se guarda en `acumulado_reviews` — una lista que puede tener cientos de elementos y ocupar varios cientos de MB.

Después del `pd.concat()`, toda esa información ya está consolidada en `df_reviews`. La lista `acumulado_reviews` quedó duplicando esa memoria sin ningún uso.

`del acumulado_reviews` elimina la única referencia que Python tiene a esa lista. Al no quedar ninguna variable apuntando a ella, Python libera la memoria **en ese mismo momento** — no hay que esperar ni hacer nada más.

```
Antes del del:
  acumulado_reviews  →  [chunk_0, chunk_1, ..., chunk_N]   ~400 MB
  df_reviews         →  DataFrame consolidado              ~400 MB
                                                           ────────
                                                           ~800 MB

Después del del:
  df_reviews         →  DataFrame consolidado              ~400 MB
```

In [ ]:
COLS_REVIEW = ['review_id', 'business_id', 'user_id', 'stars', 'date']
CHUNK_SIZE  = 100_000   # chunks más grandes para un archivo de 5 GB

acumulado_reviews = []
total_leidas      = 0

for chunk in pd.read_json(REVIEW_PATH, lines=True, chunksize=CHUNK_SIZE):
    total_leidas += len(chunk)

    # 1. Proyección
    chunk = chunk[COLS_REVIEW]

    # 2. Optimización de tipos: stars va de 1 a 5 — int8 alcanza (1 byte vs 8)
    chunk['stars'] = chunk['stars'].astype('int8')

    # 3. Filtrado cruzado: solo reviews de negocios de Florida
    # Al usar un set ('ids_florida'), Pandas ejecuta la búsqueda de inmediato sin tener que convertir la estructura a hash en cada iteración del bucle.
    chunk = chunk[chunk['business_id'].isin(ids_florida)]

    if len(chunk) > 0:
        acumulado_reviews.append(chunk)

    # Progreso cada 1M de filas
    if total_leidas % 1_000_000 == 0:
        print(f"  Leídas: {total_leidas/1_000_000:.0f}M filas — acumuladas FL: {sum(len(c) for c in acumulado_reviews):,}")

df_reviews = pd.concat(acumulado_reviews, ignore_index=True)
df_reviews['date'] = pd.to_datetime(df_reviews['date'])

# Liberar la lista de chunks — ya no la necesitamos
# Todo el contenido quedó consolidado en df_reviews
del acumulado_reviews

print(f"\nReviews totales en el archivo: {total_leidas:>10,}")
print(f"Reviews de FL:                 {len(df_reviews):>10,}")
print(f"Reducción:                     {(1 - len(df_reviews)/total_leidas)*100:.1f}%")
print()
df_reviews.head()

  Leídas: 1M filas — acumuladas FL: 129,731
  Leídas: 2M filas — acumuladas FL: 269,005
  Leídas: 3M filas — acumuladas FL: 406,698
  Leídas: 4M filas — acumuladas FL: 544,714
  Leídas: 5M filas — acumuladas FL: 688,531
  Leídas: 6M filas — acumuladas FL: 834,343

Reviews totales en el archivo:  6,990,280
Reviews de FL:                    986,088
Reducción:                     85.9%



,review_id,business_id,user_id,stars,date
0,rGQRf8UafX7OTlMNN19I8A,uMvVYRgGNXf5boolA9HXTw,1WHRWwQmZOZDAhp2Qyny4g,5,2015-06-21 14:48:06
1,OAhBYw8IQ6wlfw1owXWRWw,BVndHaLihEYbr76Z0CMEGw,1C2lxzUo1Hyye4RFIXly3g,5,2014-10-11 16:22:06
2,u2vzZaOqJ2feRshaaF1doQ,CLEWowfkj-wKYJlQDqT1aw,NDZvyYHTUWWu-kqgQzzDGQ,5,2016-03-07 00:02:18
3,EZarjNNbO_2yH1Xbizog9g,pR8u8hXf1vvzoAGOoKHQqQ,R_W9WlKiA56VzVbRzTULQQ,5,2016-08-25 17:17:46
4,5GGfWhVGmubQLmbVbye8vw,AgbRp5NLsP1-J1fdg6Hdcw,B6G4f3UX1Z5_CpyV1xXm8Q,5,2017-05-13 17:15:09


---
## 7. Merge entre los dos DataFrames

Ahora tenemos dos DataFrames en memoria:

| DataFrame | Filas | Columnas clave |
|-----------|-------|----------------|
| `df_business` | ~X.000 | `business_id`, `name`, `city`, `categories`, `stars` |
| `df_reviews`  | ~X.000.000 | `business_id`, `stars`, `date` |

Los unimos por `business_id` para tener, en cada review, los metadatos del negocio.

> Esto es posible porque **filtramos antes de hacer el merge**. Si hubiéramos cargado
> los 7 millones de reviews sin filtrar, el merge hubiera generado un DataFrame gigante.

In [30]:
# Merge: cada review queda asociada a los datos del negocio
# Sufijos para distinguir 'stars' de reviews vs. 'stars' de business
df = df_reviews.merge(
    df_business[['business_id', 'name', 'city', 'categories', 'stars', 'review_count']],
    on='business_id',
    how='inner',
    suffixes=('_review', '_business')
)

print(f"Filas después del merge: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
print()
df.head()

Filas después del merge: 986,088
Columnas: ['review_id', 'business_id', 'user_id', 'stars_review', 'date', 'name', 'city', 'categories', 'stars_business', 'review_count']



,review_id,business_id,user_id,stars_review,date,name,city,categories,stars_business,review_count
0,rGQRf8UafX7OTlMNN19I8A,uMvVYRgGNXf5boolA9HXTw,1WHRWwQmZOZDAhp2Qyny4g,5,2015-06-21 14:48:06,Shalimar Fine Jewelers,Tampa,"Shopping, Jewelry",2.5,8
1,OAhBYw8IQ6wlfw1owXWRWw,BVndHaLihEYbr76Z0CMEGw,1C2lxzUo1Hyye4RFIXly3g,5,2014-10-11 16:22:06,Mamas Kitchen,Tampa,"Sandwiches, Restaurants, American (New), Ameri...",4.5,162
2,u2vzZaOqJ2feRshaaF1doQ,CLEWowfkj-wKYJlQDqT1aw,NDZvyYHTUWWu-kqgQzzDGQ,5,2016-03-07 00:02:18,Blow Bar Express Styling Salon,Tampa,"Makeup Artists, Blow Dry/Out Services, Beauty ...",4.0,55
3,EZarjNNbO_2yH1Xbizog9g,pR8u8hXf1vvzoAGOoKHQqQ,R_W9WlKiA56VzVbRzTULQQ,5,2016-08-25 17:17:46,Mike's Auto Detailing,Tampa,"Auto Detailing, Car Wash, Automotive",4.5,48
4,5GGfWhVGmubQLmbVbye8vw,AgbRp5NLsP1-J1fdg6Hdcw,B6G4f3UX1Z5_CpyV1xXm8Q,5,2017-05-13 17:15:09,Hands On Bicycle,Clearwater,"Sporting Goods, Bikes, Shopping",5.0,13


In [32]:
# Resumen de memoria de los tres DataFrames
def mem_df(df, nombre):
    mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"  {nombre:<20} {len(df):>10,} filas   {mb:>8.1f} MB")

print("DataFrame            Filas              Memoria")
print("-" * 50)
mem_df(df_business, "df_business")
mem_df(df_reviews,  "df_reviews")
mem_df(df,          "df (merged)")

DataFrame            Filas              Memoria
--------------------------------------------------
  df_business              21,540 filas        3.7 MB
  df_reviews              986,088 filas       93.5 MB
  df (merged)             986,088 filas      220.0 MB


In [33]:
output_path = 'reviews_business_florida.csv'
df.to_csv(output_path, index=False)

### ¿Qué es Pickle en Python?

**Pickle** es el módulo estándar de Python utilizado para la **serialización y deserialización** de objetos. 

Permite convertir casi cualquier estructura de datos de Python (diccionarios, listas, modelos entrenados de Machine Learning o DataFrames de Pandas) en una secuencia de bytes que puede **guardarse en el disco** o **transmitirse por una red**, y luego reconstruirse exactamente igual.

---

### Conceptos Clave

* **Serialización (*Pickling*):** Es el proceso de empaquetar un objeto de Python y sus estructuras internas en un flujo de bytes binarios. Se realiza con `pickle.dump()` o con métodos integrados de Pandas como `.to_pickle()`.
* **Deserialización (*Unpickling*):** Es el proceso inverso; toma ese archivo binario de bytes y lo reconstruye en la memoria RAM como un objeto de Python listo para ser utilizado de nuevo. Se realiza con `pickle.load()` o `pd.read_pickle()`.

---

### Ventajas Principales

1. **Fidelidad Total:** A diferencia de formatos de texto plano como CSV, Pickle preserva la estructura exacta del objeto, incluyendo metadatos, índices de Pandas y tipos de datos específicos (por ejemplo, categorías o enteros optimizados).
2. **Comodidad:** No requiere transformar tus datos complejos a formatos tabulares rígidos. Puede guardar listas complejas de diccionarios, objetos personalizados o modelos de Machine Learning de una sola vez.
3. **Optimización con Pandas:** Para DataFrames grandes, leer y escribir en formato Pickle suele ser **mucho más rápido** que hacerlo en CSV, ya que Pandas no tiene que reinterpretar el tipo de cada columna línea por línea.

In [34]:
# Guardar el DataFrame resultante del merge en formato Pickle
output_path = 'reviews_business_florida.pkl'
df.to_pickle(output_path)

In [ ]:
import time

csv_path = 'reviews_business_florida.csv'
pkl_path = 'reviews_business_florida.pkl'

print("--- PRUEBA DE RENDIMIENTO: LECTURA ---")

# 1. Medir tiempo de lectura del CSV
inicio_csv = time.time()
df_csv = pd.read_csv(csv_path)
tiempo_csv = time.time() - inicio_csv

print(f"CSV leído en:     {tiempo_csv:.4f} segundos ({len(df_csv):,} filas)")

# 2. Medir tiempo de lectura del Pickle
inicio_pkl = time.time()
df_pkl = pd.read_pickle(pkl_path)
tiempo_pkl = time.time() - inicio_pkl

print(f"Pickle leído en:  {tiempo_pkl:.4f} segundos ({len(df_pkl):,} filas)")

# Comparación porcentual
if tiempo_pkl < tiempo_csv:
    diferencia = (tiempo_csv - tiempo_pkl) / tiempo_csv * 100
    print(f"\n¡El Pickle fue un {diferencia:.1f}% más rápido de leer que el CSV!")
else:
    diferencia = (tiempo_pkl - tiempo_csv) / tiempo_pkl * 100
    print(f"\nEl CSV fue un {diferencia:.1f}% más rápido de leer en este caso.")

--- PRUEBA DE RENDIMIENTO: LECTURA ---
CSV leído en:     1.8219 segundos (986,088 filas)
Pickle leído en:  0.0779 segundos (986,088 filas)

¡El Pickle fue un 95.7% más rápido de leer que el CSV!


---
## El pipeline completo

Lo que construimos paso a paso:

```
business.json (120 MB)          review.json (5 GB)
      │                                │
      │  chunksize + proyección        │  chunksize + proyección
      │  filtro: state == 'FL'         │  filtro: business_id in ids_florida
      │                                │
      ▼                                ▼
 df_business                      df_reviews
 (~X.000 filas)                  (~X.000.000 filas)
      │                                │
      └──────────── merge ─────────────┘
                      │
                      ▼
                df (merged)
         reviews de FL con datos del negocio
```

Con `df` podemos responder preguntas que cruzan datos de los dos archivos:
- ¿Cuáles categorías tienen mejor rating real (calculado desde reviews)?
- ¿El rating declarado en `business` coincide con el promedio real de reviews?
- ¿Cómo evolucionó la actividad de reviews en FL por año?
- ¿Qué ciudades concentran más volumen de reviews?